# CampusRAG: Retrieval-Augmented Generation

In this notebook, we build the RAG pipeline step by step. For now, we focus on the **retrieval** stage: converting a question into an embedding, searching Qdrant, and building the relevant context.

Overall flow: `Question → Query Embedding → Qdrant Search → Context → Prompt → LLM Answer`

## Before we start

The cells in `02_chunking.ipynb` have already embedded the chunks and stored them in Qdrant. We will not embed the chunks again here; we only load the embedding model for the query and open the Qdrant database for searching.

Use the same notebook kernel as before: `.venv-gte-old`.

In [1]:
from pathlib import Path

import numpy as np
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

d:\Projects\Python\CampusRAG\.venv-gte-old\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## 1. Load the embedding model

This is the same model used to build the vector database. The query model must be the same as the chunk model; otherwise, the vectors will not belong to the same representation space and the search results will not be meaningful.

In [2]:
MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"

embedding_model = SentenceTransformer(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Embedding model loaded")
print("Device:", embedding_model.device)

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Embedding model loaded
Device: cuda:0


## 2. Open the Qdrant collection

Qdrant stores and searches the vectors. Here, we only open the existing collection; we do not build anything again.

In [3]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

db_path = project_root / "data" / "vector_db"
collection_name = "campus_chunks"

client = QdrantClient(path=str(db_path))

print("Qdrant database:", db_path)
print("Collection:", collection_name)
print("Number of points:", client.count(collection_name).count)

Qdrant database: d:\Projects\Python\CampusRAG\data\vector_db
Collection: campus_chunks
Number of points: 71062


C:\Users\LOQ\AppData\Local\Temp\ipykernel_26104\1923545384.py:8: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <campus_chunks> contains 71062 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=str(db_path))


## 3. Convert the user question into an embedding

Qdrant does not understand the raw question text. We convert the question into a vector using the same embedding model, then compare it with the stored vectors.

In [4]:
query = "استاد حافظی چطوره؟"

query_vector = embedding_model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
).astype(np.float32)

print("Query:", query)
print("Vector shape:", query_vector.shape)
print("Vector norm:", round(float(np.linalg.norm(query_vector)), 4))

Query: استاد حافظی چطوره؟
Vector shape: (768,)
Vector norm: 1.0


## 4. Search for the most relevant chunks

At this stage, we retrieve a larger candidate set. The `score` measures semantic similarity; it is not a percentage. Later, we will rerank these candidates with a small freshness boost.

In [5]:
candidate_k = 50

search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=candidate_k,
    with_payload=True,
).points

for rank, result in enumerate(search_results[:5], start=1):
    print(f"{rank}. Score: {result.score:.4f}")
    print(result.payload.get("text", ""))
    print("-" * 60)

1. Score: 0.8081
[message 30153 | 2025-09-24T11:31:06+00:00] حافظی چطوره؟
------------------------------------------------------------
2. Score: 0.8007
[message 6165 | 2023-10-25T06:45:33+00:00] کسی راه ارتباطی با استاد علی حافظی داره ؟
------------------------------------------------------------
3. Score: 0.7681
[message 72349 | 2025-04-09T04:38:30+00:00] آقا کسی ازین کلاس شی گرایی علی حافظی خبر نداره ؟
------------------------------------------------------------
4. Score: 0.7538
[message 122976 | 2025-09-28T19:06:41+00:00] دوستان استاد علی حافظی میشناسید؟ برداشتید تاحالا؟
------------------------------------------------------------
5. Score: 0.7320
[message 121869 | 2025-09-24T14:50:01+00:00] سلام
علی حافظی میشناسه کسی؟
------------------------------------------------------------


## 5. Inspect source metadata

The vector is used for ranking, but the payload contains the information needed for a useful answer: source name, message date, message ID, and media paths. The payload is where retrieval becomes traceable.

In [6]:
first_payload = search_results[0].payload
metadata = first_payload.get("metadata", {})

print("Payload keys:", list(first_payload.keys()))
print("Source name:", metadata.get("source_name"))
print("Source ID:", metadata.get("source_id"))
print("Message date:", metadata.get("message_date"))
print("Message IDs:", metadata.get("message_ids"))
print("Media paths:", metadata.get("media_paths"))

Payload keys: ['text', 'metadata']
Source name: گروه استاد بانک
Source ID: -1001769812584
Message date: 2025-09-24T11:31:06+00:00
Message IDs: [30153]
Media paths: []


## 6. Convert metadata into dates, links, and image sources

A thread can contain several messages, so we first try to use the per-message `message_sources` list. The fallback keeps older vector records usable when they only contain `message_ids`, `message_date`, and `media_paths`.

In [7]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

TEHRAN_TIMEZONE = ZoneInfo("Asia/Tehran")

def format_date(value):
    if not value:
        return "Unknown date"

    try:
        parsed = datetime.fromisoformat(
            str(value).replace("Z", "+00:00")
        )
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)

        local_time = parsed.astimezone(TEHRAN_TIMEZONE)
        age_days = max(
            0,
            (datetime.now(timezone.utc) - parsed.astimezone(timezone.utc)).days,
        )

        if age_days < 30:
            age = f"{age_days} days ago"
        elif age_days < 365:
            age = f"{age_days // 30} months ago"
        else:
            age = f"{age_days // 365} years ago"

        return f"{local_time:%Y-%m-%d %H:%M} ({age})"
    except (TypeError, ValueError):
        return str(value)

def build_message_link(source_id, message_id, source_username=None):
    if message_id is None:
        return None

    username = str(source_username or "").strip().lstrip("@")
    if username:
        return f"https://t.me/{username}/{message_id}"

    source_id = str(source_id or "")
    if source_id.startswith("-100"):
        return f"https://t.me/c/{source_id[4:]}/{message_id}"

    return None

def get_message_sources(metadata):
    sources = metadata.get("message_sources")
    if isinstance(sources, list) and sources:
        return sources

    message_ids = metadata.get("message_ids") or []
    media_paths = metadata.get("media_paths") or []
    message_date = metadata.get("message_date")

    return [
        {
            "message_id": message_id,
            "message_date": message_date,
            "media_path": media_paths[index]
            if index < len(media_paths)
            else None,
        }
        for index, message_id in enumerate(message_ids)
    ]

def get_result_sources(result):
    metadata = result.payload.get("metadata", {})
    return [
        {
            "message_id": source.get("message_id"),
            "date": format_date(source.get("message_date")),
            "link": build_message_link(
                metadata.get("source_id"),
                source.get("message_id"),
                metadata.get("source_username"),
            ),
            "media_path": source.get("media_path"),
        }
        for source in get_message_sources(metadata)
    ]

## 7. Rerank candidates by freshness

Recency is a second signal, not a replacement for semantic similarity. We first retrieve 50 semantically similar candidates, then give newer messages a configurable bonus. `half_life_days` controls how quickly old messages lose freshness, while `freshness_weight` controls how important recency is.

In [8]:
def parse_datetime(value):
    if not value:
        return None

    try:
        parsed = datetime.fromisoformat(
            str(value).replace("Z", "+00:00")
        )
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)
        return parsed.astimezone(timezone.utc)
    except (TypeError, ValueError):
        return None

def get_result_latest_datetime(result):
    metadata = result.payload.get("metadata", {})
    dates = [
        source.get("message_date")
        for source in get_message_sources(metadata)
    ]
    dates.append(metadata.get("message_date"))
    parsed_dates = [parse_datetime(value) for value in dates]
    parsed_dates = [value for value in parsed_dates if value is not None]
    return max(parsed_dates) if parsed_dates else None

def freshness_score(result, now=None, half_life_days=180):
    latest = get_result_latest_datetime(result)
    if latest is None:
        return 0.0

    now = now or datetime.now(timezone.utc)
    age_days = max(0.0, (now - latest).total_seconds() / 86400)
    return 2 ** (-age_days / half_life_days)

def rerank_with_freshness(
    results,
    freshness_weight=0.20,
    half_life_days=180,
):
    if not results:
        return []

    ranked = []
    for result in results:
        semantic_score = float(result.score)
        # Qdrant uses cosine similarity here, so map [-1, 1] to [0, 1].
        semantic_normalized = (semantic_score + 1.0) / 2.0
        fresh = freshness_score(result, half_life_days=half_life_days)
        combined = (1 - freshness_weight) * semantic_normalized
        combined += freshness_weight * fresh
        ranked.append((combined, semantic_score, fresh, result))

    ranked.sort(key=lambda item: item[0], reverse=True)

    for rank, (combined, semantic, fresh, result) in enumerate(
        ranked[:5],
        start=1,
    ):
        latest = get_result_latest_datetime(result)
        print(
            f"{rank}. combined={combined:.4f} | "
            f"semantic={semantic:.4f} | freshness={fresh:.4f} | "
            f"date={format_date(latest.isoformat() if latest else None)}"
        )

    return [item[3] for item in ranked]

search_results = rerank_with_freshness(
    search_results,
    freshness_weight=0.20,
    half_life_days=180,
)[:5]

1. combined=0.8748 | semantic=0.7083 | freshness=0.9576 | date=2026-10-11 15:58 (11 days ago)
2. combined=0.8704 | semantic=0.6968 | freshness=0.9583 | date=2026-10-11 20:22 (11 days ago)
3. combined=0.8455 | semantic=0.6872 | freshness=0.8532 | date=2026-09-11 16:20 (1 months ago)
4. combined=0.7672 | semantic=0.8081 | freshness=0.2199 | date=2025-09-24 15:01 (1 years ago)
5. combined=0.7593 | semantic=0.7209 | freshness=0.3547 | date=2026-01-26 17:23 (8 months ago)


## 8. Build the context

The language model should not see the entire database. We place only these top results inside the prompt as context. Each source now carries its date and Telegram link so the answer can be checked later.

In [9]:
def build_context(results):
    context_parts = []

    for source_number, result in enumerate(results, start=1):
        payload = result.payload
        metadata = payload.get("metadata", {})
        source_name = metadata.get("source_name", "Unknown source")
        source_details = get_result_sources(result)
        dates = ", ".join(
            source["date"] for source in source_details
        )
        links = ", ".join(
            source["link"]
            for source in source_details
            if source["link"]
        )

        context_parts.append(
            f"[Source {source_number}]\n"
            f"Source: {source_name}\n"
            f"Date(s): {dates or 'Unknown date'}\n"
            f"Telegram link(s): {links or 'Unavailable'}\n"
            f"Text:\n{payload.get('text', '')}"
        )

    return "\n\n".join(context_parts)

context = build_context(search_results)
print(context)

[Source 1]
Source: گروه دانشجویی شمسی پور
Date(s): 2026-10-11 12:47 (11 days ago), 2026-10-11 12:47 (11 days ago), 2026-10-11 13:24 (11 days ago), 2026-10-11 15:27 (11 days ago), 2026-10-11 15:30 (11 days ago), 2026-10-11 15:57 (11 days ago), 2026-10-11 15:58 (11 days ago)
Telegram link(s): https://t.me/stvc_gap/162572, https://t.me/stvc_gap/162573, https://t.me/stvc_gap/162575, https://t.me/stvc_gap/162583, https://t.me/stvc_gap/162584, https://t.me/stvc_gap/162586, https://t.me/stvc_gap/162587
Text:
[message 162572 | 2026-10-11T09:17:23+00:00] 📜✨ **به مناسبت روز بزرگداشت حافظ**

اداره امور فرهنگی و اجتماعی با همکاری **کانون شعر و ادب، کانون موسیقی دلشدگان و انجمن علمی کامپیوتر دانشکده شهید شمسی‌پور** برگزار می‌نماید:

🌿 **نشست فرهنگی «از مولانا تا حافظ»** 
با موضوع: 
📖 **وجوه اهمیت حافظ در فرهنگ ایرانی**

🎙 سخنران: 
**جناب آقای دکتر حمید ده‌نبی** 
عضو هیأت علمی دانشکده شهید شمسی‌پور

🗓 دوشنبه ۲۰ مهر ۱۴۰۵ 
⏰ ساعت ۱۰ الی ۱۲ 
📍 سالن اجتماعات دانشکده

🎶 همراه با اجرای **موسیقی زنده توسط 

## 9. Prepare sources for the user interface

The answer should expose its evidence. This cell creates readable source cards and collects existing image files for the side panel.

In [10]:
def build_sources_markdown(results):
    parts = ["### Retrieved sources"]

    for index, result in enumerate(results, start=1):
        metadata = result.payload.get("metadata", {})
        source_name = metadata.get("source_name", "Unknown source")
        parts.append(
            f"**Source {index} — {source_name} — score: {result.score:.4f}**"
        )

        for source in get_result_sources(result):
            line = f"- Date: {source['date']}"
            if source["link"]:
                line += f" — [Open Telegram message]({source['link']})"
            parts.append(line)

        text = result.payload.get("text", "").strip()
        parts.append(f"> {text.replace(chr(10), chr(10) + '> ')}")

    return "\n\n".join(parts)

def get_image_gallery_items(results):
    gallery_items = []
    seen_paths = set()

    for index, result in enumerate(results, start=1):
        for source in get_result_sources(result):
            media_path = source["media_path"]
            if not media_path:
                continue

            raw_path = Path(media_path)
            candidates = [raw_path]
            if not raw_path.is_absolute():
                candidates.append(project_root / raw_path)
            path = next(
                (candidate for candidate in candidates if candidate.exists()),
                candidates[-1],
            )
            path = path.resolve()
            if not path.exists() or path in seen_paths:
                continue

            seen_paths.add(path)
            gallery_items.append(
                (str(path), f"Source {index} — {source['date']}")
            )

    return gallery_items

sources_markdown = build_sources_markdown(search_results)
image_gallery_items = get_image_gallery_items(search_results)

print(sources_markdown)
print("Number of images:", len(image_gallery_items))

### Retrieved sources

**Source 1 — گروه دانشجویی شمسی پور — score: 0.7083**

- Date: 2026-10-11 12:47 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162572)

- Date: 2026-10-11 12:47 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162573)

- Date: 2026-10-11 13:24 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162575)

- Date: 2026-10-11 15:27 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162583)

- Date: 2026-10-11 15:30 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162584)

- Date: 2026-10-11 15:57 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162586)

- Date: 2026-10-11 15:58 (11 days ago) — [Open Telegram message](https://t.me/stvc_gap/162587)

> [message 162572 | 2026-10-11T09:17:23+00:00] 📜✨ **به مناسبت روز بزرگداشت حافظ**
> 
> اداره امور فرهنگی و اجتماعی با همکاری **کانون شعر و ادب، کانون موسیقی دلشدگان و انجمن علمی کامپیوتر دانشکده شهید شمسی‌پور** برگزار می‌نماید:
> 
> 🌿 **نشست فرهنگی «

## 10. Build the prompt

We do not call the language model yet. We only build the prompt so we can inspect exactly what will be sent to the LLM.

Important rule: the model must answer only from the context and must not guess when the available information is insufficient.

In [11]:
SYSTEM_PROMPT = """
You are a helpful university information assistant.
Answer in Persian.
Use only the information inside the provided context.
If the context is not sufficient, say that you do not have enough information.
Do not invent names, dates, courses, or opinions.
You can use the people idea if they say a bad word about someone just say it DO NOT CENSORE.
When possible, mention the source number used for the answer.
""".strip()

def build_prompt(query, context, conversation_history=""):
    return f"""
{SYSTEM_PROMPT}

Context:
{context}

Conversation history:
{conversation_history or 'No previous conversation.'}

User question:
{query}

Answer in Persian:
""".strip()

prompt = build_prompt(query, context)
print(prompt)

You are a helpful university information assistant.
Answer in Persian.
Use only the information inside the provided context.
If the context is not sufficient, say that you do not have enough information.
Do not invent names, dates, courses, or opinions.
You can use the people idea if they say a bad word about someone just say it DO NOT CENSORE.
When possible, mention the source number used for the answer.

Context:
[Source 1]
Source: گروه دانشجویی شمسی پور
Date(s): 2026-10-11 12:47 (11 days ago), 2026-10-11 12:47 (11 days ago), 2026-10-11 13:24 (11 days ago), 2026-10-11 15:27 (11 days ago), 2026-10-11 15:30 (11 days ago), 2026-10-11 15:57 (11 days ago), 2026-10-11 15:58 (11 days ago)
Telegram link(s): https://t.me/stvc_gap/162572, https://t.me/stvc_gap/162573, https://t.me/stvc_gap/162575, https://t.me/stvc_gap/162583, https://t.me/stvc_gap/162584, https://t.me/stvc_gap/162586, https://t.me/stvc_gap/162587
Text:
[message 162572 | 2026-10-11T09:17:23+00:00] 📜✨ **به مناسبت روز بزرگداشت ح

## 11. Build a retrieval UI with Gradio

This interface behaves like a chatbot: it retrieves the best chunks, builds a grounded prompt, and sends it to the local Ollama model `gemma4:e2b`. The chat shows only the answer; sources and images stay in the right sidebar.

Run this cell last. `demo.launch()` starts a local web server and keeps the notebook cell busy until you interrupt it.

In [ ]:
import gradio as gr
from openai import OpenAI

BASE_URL = "http://127.0.0.1:11434/v1"
MODEL = "gemma4:e2b"
ollama = OpenAI(api_key="ollama", base_url=BASE_URL)

# Run this cell after the retrieval and prompt exercises above.

def format_chat_history(history, max_messages=6):
    messages = []
    for message in (history or [])[-max_messages:]:
        if not isinstance(message, dict):
            continue
        role = message.get("role")
        content = str(message.get("content", "")).strip()
        if role in {"user", "assistant"} and content:
            messages.append(f"{role}: {content}")
    return "\n".join(messages)

def build_retrieval_query(user_query, history):
    previous_questions = [
        str(message.get("content", "")).strip()
        for message in (history or [])
        if isinstance(message, dict)
        and message.get("role") == "user"
        and message.get("content")
    ][-2:]
    return "\n".join(previous_questions + [user_query])

def retrieve_for_ui(user_query, history):
    history = history or []
    user_query = (user_query or "").strip()

    if not user_query:
        return history, [], "Please enter a question.", user_query

    retrieval_query = build_retrieval_query(user_query, history)

    user_query_vector = embedding_model.encode(
        retrieval_query,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype(np.float32)

    results = client.query_points(
        collection_name=collection_name,
        query=user_query_vector.tolist(),
        limit=candidate_k,
        with_payload=True,
    ).points
    results = rerank_with_freshness(
        results,
        freshness_weight=0.20,
        half_life_days=180,
    )[:10]

    context = build_context(results)
    prompt = build_prompt(
        user_query,
        context,
        format_chat_history(history),
    )
    evidence = build_sources_markdown(results)

    try:
        completion = ollama.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
        )
        answer = completion.choices[0].message.content
    except Exception as error:
        answer = (
            f"Generation failed: {error}\n\n"
            "Make sure Ollama is running and gemma4:e2b is installed."
        )

    response = answer or "The model returned an empty answer."

    history = history + [
        {"role": "user", "content": user_query},
        {"role": "assistant", "content": response},
    ]

    return history, get_image_gallery_items(results), evidence, ""

with gr.Blocks(title="CampusRAG") as demo:
    gr.Markdown("# CampusRAG\nDense retrieval with Qdrant")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=600)
            with gr.Row():
                query_box = gr.Textbox(
                    label="Question",
                    placeholder="Ask about a professor, course, or announcement...",
                    scale=5,
                )
                send_button = gr.Button("Send", variant="primary", scale=1)

        with gr.Column(scale=1):
            gallery = gr.Gallery(
                label="Images from retrieved messages",
                columns=1,
                height=600,
                object_fit="contain",
            )
            sources_output = gr.Markdown(label="Sources")

    outputs = [chatbot, gallery, sources_output, query_box]
    send_button.click(
        retrieve_for_ui,
        inputs=[query_box, chatbot],
        outputs=outputs,
    )
    query_box.submit(
        retrieve_for_ui,
        inputs=[query_box, chatbot],
        outputs=outputs,
    )

demo.launch(
    allowed_paths=[str(project_root / "data" / "media")]
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


1. combined=0.9448 | semantic=0.8816 | freshness=0.9610 | date=2026-10-12 14:13 (10 days ago)
2. combined=0.9359 | semantic=0.8790 | freshness=0.9214 | date=2026-10-01 15:42 (21 days ago)
3. combined=0.9274 | semantic=0.8839 | freshness=0.8691 | date=2026-09-16 11:35 (1 months ago)
4. combined=0.9259 | semantic=0.8815 | freshness=0.8666 | date=2026-09-15 18:00 (1 months ago)
5. combined=0.9097 | semantic=0.8793 | freshness=0.7898 | date=2026-08-22 15:27 (2 months ago)
1. combined=0.9448 | semantic=0.8816 | freshness=0.9610 | date=2026-10-12 14:13 (10 days ago)
2. combined=0.9359 | semantic=0.8790 | freshness=0.9214 | date=2026-10-01 15:42 (21 days ago)
3. combined=0.9274 | semantic=0.8839 | freshness=0.8691 | date=2026-09-16 11:35 (1 months ago)
4. combined=0.9259 | semantic=0.8815 | freshness=0.8666 | date=2026-09-15 18:00 (1 months ago)
5. combined=0.9097 | semantic=0.8793 | freshness=0.7898 | date=2026-08-22 15:27 (2 months ago)
